# Pizza-Bot - Partie 1 : le duel d'emplacements

Deux emplacements libres : le Campus et la Gare. Pizza-Bot et Mamma-Auto décident en même
temps, sans connaître le choix de l'autre.

Matrice des marges mensuelles, en k€. Premier nombre : Pizza-Bot. Second : Mamma-Auto.

| Pizza-Bot \ Mamma-Auto | Campus | Gare |
|---|---|---|
| **Campus** | 4 ; 4 | 9 ; 7 |
| **Gare** | 6 ; 8 | 3 ; 3 |

Règles du projet : Python de base, aucun import.

## Étape 1.2 — Les meilleures réponses

In [1]:
EMPLACEMENTS = ["Campus", "Gare"]

GAINS_EMPLACEMENT = {
    ("Campus", "Campus"): (4, 4),
    ("Campus", "Gare"): (9, 7),
    ("Gare", "Campus"): (6, 8),
    ("Gare", "Gare"): (3, 3),
}

def meilleure_reponse_pizzabot(gains, choix_mamma):
    """L'emplacement qui rapporte le plus a Pizza-Bot quand Mamma-Auto joue choix_mamma."""
    # La colonne est fixee : on essaie chaque ligne et on garde la meilleure.
    meilleur = EMPLACEMENTS[0]
    for choix in EMPLACEMENTS:
        if gains[(choix, choix_mamma)][0] > gains[(meilleur, choix_mamma)][0]:
            meilleur = choix
    return meilleur

def meilleure_reponse_mamma(gains, choix_pizzabot):
    """L'emplacement qui rapporte le plus a Mamma-Auto quand Pizza-Bot joue choix_pizzabot."""
    # La ligne est fixee : on essaie chaque colonne. Le gain de Mamma-Auto est le second du couple.
    meilleur = EMPLACEMENTS[0]
    for choix in EMPLACEMENTS:
        if gains[(choix_pizzabot, choix)][1] > gains[(choix_pizzabot, meilleur)][1]:
            meilleur = choix
    return meilleur

for c in EMPLACEMENTS:
    print(f"Si Mamma-Auto joue {c:6}, Pizza-Bot répond {meilleure_reponse_pizzabot(GAINS_EMPLACEMENT, c)}")
for c in EMPLACEMENTS:
    print(f"Si Pizza-Bot joue {c:6}, Mamma-Auto répond {meilleure_reponse_mamma(GAINS_EMPLACEMENT, c)}")

# --- Vérification : ne rien modifier sous cette ligne ---
dilemme = {("Campus", "Campus"): (3, 3), ("Campus", "Gare"): (0, 5),
           ("Gare", "Campus"): (5, 0), ("Gare", "Gare"): (1, 1)}
assert meilleure_reponse_pizzabot(dilemme, "Campus") == "Gare", "face à Campus, 5 vaut mieux que 3"
assert meilleure_reponse_pizzabot(dilemme, "Gare") == "Gare", "face à Gare, 1 vaut mieux que 0"
assert meilleure_reponse_mamma(dilemme, "Campus") == "Gare", "le gain de Mamma-Auto est le SECOND du couple"
assert meilleure_reponse_mamma(dilemme, "Gare") == "Gare"
print("Meilleures réponses validées")

Si Mamma-Auto joue Campus, Pizza-Bot répond Gare
Si Mamma-Auto joue Gare  , Pizza-Bot répond Campus
Si Pizza-Bot joue Campus, Mamma-Auto répond Gare
Si Pizza-Bot joue Gare  , Mamma-Auto répond Campus
Meilleures réponses validées


**Lecture du résultat.** Chaque joueur veut aller là où l'autre n'est pas : face au Campus,
la meilleure réponse est la Gare, et face à la Gare, c'est le Campus. Aucun des deux n'a de
stratégie dominante, puisque sa meilleure réponse change selon le choix de l'adversaire.
Les quatre réponses correspondent aux soulignements du brouillon.

## Étape 1.3 - Le test de la déviation

In [2]:
EMPLACEMENTS = ["Campus", "Gare"]

GAINS_EMPLACEMENT = {
    ("Campus", "Campus"): (4, 4),
    ("Campus", "Gare"): (9, 7),
    ("Gare", "Campus"): (6, 8),
    ("Gare", "Gare"): (3, 3),
}

def est_equilibre_nash(gains, choix_pizzabot, choix_mamma):
    """True si aucun joueur ne gagne STRICTEMENT plus en changeant SEUL de strategie."""
    gain_pizzabot = gains[(choix_pizzabot, choix_mamma)][0]
    gain_mamma = gains[(choix_pizzabot, choix_mamma)][1]

    # Deviations de Pizza-Bot : la colonne de Mamma-Auto ne bouge pas.
    for autre in EMPLACEMENTS:
        if gains[(autre, choix_mamma)][0] > gain_pizzabot:
            return False

    # Deviations de Mamma-Auto : la ligne de Pizza-Bot ne bouge pas.
    for autre in EMPLACEMENTS:
        if gains[(choix_pizzabot, autre)][1] > gain_mamma:
            return False

    return True

def trouver_equilibres_nash(gains):
    """La liste de toutes les cases (choix_pizzabot, choix_mamma) qui sont des equilibres."""
    equilibres = []
    for choix_pizzabot in EMPLACEMENTS:
        for choix_mamma in EMPLACEMENTS:
            if est_equilibre_nash(gains, choix_pizzabot, choix_mamma):
                equilibres.append((choix_pizzabot, choix_mamma))
    return equilibres

for case in trouver_equilibres_nash(GAINS_EMPLACEMENT):
    print(f"Équilibre de Nash : {case} -> gains {GAINS_EMPLACEMENT[case]}")

# --- Vérification : ne rien modifier sous cette ligne ---
dilemme = {("Campus", "Campus"): (3, 3), ("Campus", "Gare"): (0, 5),
           ("Gare", "Campus"): (5, 0), ("Gare", "Gare"): (1, 1)}
assert est_equilibre_nash(dilemme, "Campus", "Campus") is False, "chacun gagne à passer à la Gare"
assert est_equilibre_nash(dilemme, "Gare", "Gare") is True
assert trouver_equilibres_nash(dilemme) == [("Gare", "Gare")], "un seul équilibre dans le dilemme"
indifferent = {("Campus", "Campus"): (2, 2), ("Campus", "Gare"): (2, 2),
               ("Gare", "Campus"): (2, 2), ("Gare", "Gare"): (2, 2)}
assert len(trouver_equilibres_nash(indifferent)) == 4, "tout rapporte 2 : les 4 cases sont des équilibres"
print("Équilibres validés")

Équilibre de Nash : ('Campus', 'Gare') -> gains (9, 7)
Équilibre de Nash : ('Gare', 'Campus') -> gains (6, 8)
Équilibres validés


**Lecture du résultat.** Le programme trouve deux équilibres : (Campus ; Gare) avec des
gains de 9 et 7, et (Gare ; Campus) avec 6 et 8. Ce sont exactement les deux cases doublement
soulignées du brouillon. La fonction compare des gains, et non des noms de stratégies : c'est
ce qui lui permet de trouver les quatre équilibres de la matrice où tout rapporte 2.

## Bonus - L'équilibre en stratégies mixtes

Chacun tire son emplacement au sort. On cherche la probabilité qui rend l'adversaire
indifférent entre ses deux stratégies.

In [3]:
EMPLACEMENTS = ["Campus", "Gare"]

GAINS_EMPLACEMENT = {
    ("Campus", "Campus"): (4, 4),
    ("Campus", "Gare"): (9, 7),
    ("Gare", "Campus"): (6, 8),
    ("Gare", "Gare"): (3, 3),
}

def gain_moyen_mamma(gains, q, choix_mamma):
    """Ce que gagne Mamma-Auto en moyenne si Pizza-Bot joue Campus avec la probabilite q."""
    gain_si_campus = gains[("Campus", choix_mamma)][1]
    gain_si_gare = gains[("Gare", choix_mamma)][1]
    return q * gain_si_campus + (1 - q) * gain_si_gare

def gain_moyen_pizzabot(gains, p, choix_pizzabot):
    """Ce que gagne Pizza-Bot en moyenne si Mamma-Auto joue Campus avec la probabilite p."""
    gain_si_campus = gains[(choix_pizzabot, "Campus")][0]
    gain_si_gare = gains[(choix_pizzabot, "Gare")][0]
    return p * gain_si_campus + (1 - p) * gain_si_gare

# On balaie les probabilites de 0 a 1 par pas de 1/1000 et on garde celle qui egalise le mieux.
meilleur_q = 0
ecart_q = None
for i in range(1001):
    q = i / 1000
    ecart = abs(gain_moyen_mamma(GAINS_EMPLACEMENT, q, "Campus") - gain_moyen_mamma(GAINS_EMPLACEMENT, q, "Gare"))
    if ecart_q is None or ecart < ecart_q:
        ecart_q = ecart
        meilleur_q = q

meilleur_p = 0
ecart_p = None
for i in range(1001):
    p = i / 1000
    ecart = abs(gain_moyen_pizzabot(GAINS_EMPLACEMENT, p, "Campus") - gain_moyen_pizzabot(GAINS_EMPLACEMENT, p, "Gare"))
    if ecart_p is None or ecart < ecart_p:
        ecart_p = ecart
        meilleur_p = p

print(f"Pizza-Bot joue Campus avec q = {meilleur_q}")
print(f"Mamma-Auto joue Campus avec p = {meilleur_p}")

Pizza-Bot joue Campus avec q = 0.625
Mamma-Auto joue Campus avec p = 0.75


**Lecture du résultat.** Pizza-Bot doit jouer le Campus avec une probabilité de 0,625,
soit 5/8, et Mamma-Auto avec 0,75, soit 3/4. Ces valeurs confirment le calcul du brouillon :
8 − 4q = 3 + 4q donne q = 5/8, et 9 − 5p = 3 + 3p donne p = 3/4.